# 05 — Autoencoder za Anomaly Detection (PyTorch)
**Cilj:** Koristiti Autoencoder kao **unsupervised** pristup detekciji prevara.

**Ideja:** Autoencoder treniramo **samo na legitimnim transakcijama**. Mreža uči da rekonstruiše normalne obrasce. Kada naiđe na fraud transakciju (anomaliju), rekonstrukcijska greška će biti veća → koristimo tu grešku kao anomaly score.

$$\text{AnomalyScore}(x) = ||x - \hat{x}||^2 \quad \text{(MSE rekonstrukcija)}$$

> **Framework:** Ovaj notebook koristi **PyTorch** umesto TensorFlow/Keras.

---

## 0. Imports

Isti set biblioteka kao u prethodnim notebookovima, plus `sklearn.manifold.TSNE` za vizualizaciju latentnog prostora.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import json, os, warnings

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix,
    f1_score, roc_curve, precision_recall_curve
)
from sklearn.manifold import TSNE

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PROC    = '../data/processed/'
MODELS  = '../models/'
RESULTS = '../results/'

print(f'PyTorch: {torch.__version__} | Uređaj: {DEVICE}')

## 1. Učitavanje podataka

**Ključna specifičnost autoencodera:** Treniramo ga **isključivo na legitimnim transakcijama** (`y == 0`). Mreža nikada ne "vidi" fraud primere tokom treniranja — uči isključivo normalni obrazac. Ovo je suština unsupervised anomaly detection-a.

Validacioni i test set ostaju nepromenjeni (sadrže i legitimne i fraud transakcije) — koristimo ih samo za evaluaciju anomaly score-a.

In [ ]:
X_train = np.load(f'{PROC}X_train.npy')
X_val   = np.load(f'{PROC}X_val.npy')
X_test  = np.load(f'{PROC}X_test.npy')
y_train = np.load(f'{PROC}y_train.npy')
y_val   = np.load(f'{PROC}y_val.npy')
y_test  = np.load(f'{PROC}y_test.npy')

INPUT_DIM = X_train.shape[1]

# Trening AE samo na legitimnim transakcijama
X_train_legit = X_train[y_train == 0]
X_val_legit   = X_val[y_val == 0]

print(f'Input dim: {INPUT_DIM}')
print(f'Train (samo legitimne): {X_train_legit.shape}')
print(f'Val   (samo legitimne): {X_val_legit.shape}')
print(f'Test  (sve):            {X_test.shape} | Fraud: {y_test.sum()}')


def make_reconstruction_loader(X, batch_size=256, shuffle=True):
    """DataLoader za autoencoder: X -> X (rekonstrukcija, nema labela)."""
    X_t = torch.tensor(X, dtype=torch.float32)
    # TensorDataset sa jednim tenzarom: input i target su isti
    return DataLoader(TensorDataset(X_t), batch_size=batch_size, shuffle=shuffle)


train_legit_loader = make_reconstruction_loader(X_train_legit)
val_legit_loader   = make_reconstruction_loader(X_val_legit, shuffle=False)

# Fiksni tenzori za evaluaciju
X_val_t  = torch.tensor(X_val,  dtype=torch.float32).to(DEVICE)
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(DEVICE)

## 2. Arhitektura Autoencodera

```
Input(31) → Encoder: 64→32→16→8 (bottleneck) → Decoder: 8→16→32→64 → Output(31)
```

**Komponente:**

- **Encoder** — kompresuje ulaz u kompaktnu reprezentaciju (latentni prostor). Svaki sloj smanjuje dimenzionalnost.
- **Bottleneck (latentni prostor)** — najuža tačka mreže (8 dimenzija). Primorava mrežu da sačuva samo najvažnije informacije o normalnom ponašanju transakcija.
- **Decoder** — simetrična ogledalna kopija encodera. Rekonstruiše originalni ulaz iz latentne reprezentacije.

**PyTorch implementacija:** Razdvajamo `Encoder` i `Decoder` kao zasebne `nn.Module` klase, a `Autoencoder` ih komponuje. Ovo nam omogućava da koristimo samo encoder za ekstrakciju latentnih reprezentacija (npr. za t-SNE vizualizaciju).

**Izlaz dekodera** je linearan (bez aktivacije) — rekonstruišemo standardizovane vrednosti koje mogu biti i negativne.

In [ ]:
class Encoder(nn.Module):
    """Enkoder: kompresuje ulaz u latentni prostor."""
    def __init__(self, input_dim, hidden_units=(64, 32, 16), encoding_dim=8, dropout_rate=0.2):
        super().__init__()
        layers_list = []
        in_f = input_dim
        for units in hidden_units:
            layers_list += [
                nn.Linear(in_f, units),
                nn.BatchNorm1d(units),
                nn.ReLU(),
                nn.Dropout(p=dropout_rate)
            ]
            in_f = units
        # Bottleneck bez aktivacije — linearna projekcija
        layers_list.append(nn.Linear(in_f, encoding_dim))
        self.network = nn.Sequential(*layers_list)

    def forward(self, x):
        return self.network(x)


class Decoder(nn.Module):
    """Dekoder: rekonstruiše ulaz iz latentne reprezentacije."""
    def __init__(self, encoding_dim=8, hidden_units=(16, 32, 64), output_dim=31, dropout_rate=0.2):
        super().__init__()
        layers_list = []
        in_f = encoding_dim
        for units in hidden_units:
            layers_list += [
                nn.Linear(in_f, units),
                nn.BatchNorm1d(units),
                nn.ReLU(),
                nn.Dropout(p=dropout_rate)
            ]
            in_f = units
        # Izlaz: linearan, rekonstruiše originalne (standardizovane) vrednosti
        layers_list.append(nn.Linear(in_f, output_dim))
        self.network = nn.Sequential(*layers_list)

    def forward(self, z):
        return self.network(z)


class Autoencoder(nn.Module):
    """Kompozicija Encodera i Decodera."""
    def __init__(self, input_dim, encoding_dim=8,
                 hidden_units=(64, 32, 16), dropout_rate=0.2):
        super().__init__()
        self.encoder = Encoder(input_dim, hidden_units, encoding_dim, dropout_rate)
        # Decoder koristi obrnuti redosled hidden_units
        self.decoder = Decoder(encoding_dim, tuple(reversed(hidden_units)),
                               input_dim, dropout_rate)

    def forward(self, x):
        z = self.encoder(x)    # Kompresija
        return self.decoder(z) # Rekonstrukcija


autoencoder = Autoencoder(INPUT_DIM, encoding_dim=8)
print(autoencoder)
print(f'Ukupno parametara: {sum(p.numel() for p in autoencoder.parameters()):,}')

## 3. Treniranje Autoencodera

**Loss funkcija:** `MSELoss` (Mean Squared Error) između originalnog ulaza i rekonstrukcije. Cilj mreže je da minimizuje rekonstrukcijsku grešku za legitimne transakcije.

**Bitna razlika od klasifikatora:** U DataLoader-u imamo samo jedan tenzor (X), a i ulaz i ciljni izlaz su isti (`X_batch → X_batch`).

Pratimo val loss na `X_val_legit` (ne na celom val setu!) da ne bismo "kažnjavali" model za loše rekonstruisanje fraud uzoraka — to je upravo ono što želimo.

In [ ]:
def train_autoencoder(model, train_loader, X_val_legit_t,
                      epochs=150, lr=1e-3, patience=10):
    """
    Training autoencoder-a sa MSE loss-om.
    Ulaz i ciljni izlaz su isti (rekonstrukcija).
    Validation se prati na legitimnim uzorcima.
    """
    model = model.to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6, verbose=True
    )

    best_val_loss = float('inf')
    best_weights  = None
    patience_cnt  = 0
    train_losses, val_losses = [], []

    for epoch in range(1, epochs + 1):
        # ---------- TRENING ----------
        model.train()
        run_loss = 0.0
        for (X_batch,) in train_loader:  # TensorDataset sa jednim tenzarom
            X_batch = X_batch.to(DEVICE)
            optimizer.zero_grad()
            X_recon = model(X_batch)
            loss    = criterion(X_recon, X_batch)  # Rekonstrukcija ↔ Original
            loss.backward()
            optimizer.step()
            run_loss += loss.item() * X_batch.size(0)

        train_loss = run_loss / len(train_loader.dataset)

        # ---------- VALIDACIJA (samo legitimni uzorci!) ----------
        model.eval()
        with torch.no_grad():
            X_val_recon = model(X_val_legit_t)
            val_loss    = criterion(X_val_recon, X_val_legit_t).item()

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        scheduler.step(val_loss)

        if epoch % 10 == 0:
            print(f'Epoha {epoch:3d} | Train MSE: {train_loss:.5f} | Val MSE: {val_loss:.5f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights  = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            torch.save(best_weights, f'{MODELS}autoencoder.pt')
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= patience:
                print(f'\nEarly stopping (epoha {epoch}) | Beste Val MSE: {best_val_loss:.5f}')
                break

    model.load_state_dict(best_weights)
    return model, train_losses, val_losses


# Fiksni tenzori za legitimne uzorke
X_val_legit_t = torch.tensor(X_val_legit, dtype=torch.float32).to(DEVICE)

autoencoder, train_losses_ae, val_losses_ae = train_autoencoder(
    autoencoder, train_legit_loader, X_val_legit_t, epochs=150, patience=10
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses_ae, label='Train MSE')
ax.plot(val_losses_ae,   label='Val MSE')
ax.set_title('Autoencoder — Rekonstrukcijska Greška (MSE)')
ax.set_xlabel('Epoha')
ax.set_ylabel('MSE Loss')
ax.legend()
plt.savefig(f'{RESULTS}05_ae_history.png', bbox_inches='tight')
plt.show()

## 4. Anomaly Score = Rekonstrukcijska Greška

**Princip detekcije:** Autoencoder je naučio normalnu distribuciju transakcija. Za svaki novi uzorak računamo MSE između originalnih vrednosti i rekonstrukcije:

$$\text{AnomalyScore}(x_i) = \frac{1}{d} \sum_{j=1}^{d} (x_{ij} - \hat{x}_{ij})^2$$

- **Legitimna transakcija** → mala greška (mreža je naučila ovaj obrazac)
- **Fraud transakcija** → veća greška (anomalni obrazac koji mreža nije videla)

**Vizualizacija** distribucije grešaka po klasama nam govori koliko dobro autoencoder razdvaja dve klase — ovo je analog ROC krivoj za threshold-based decision.

In [ ]:
def reconstruction_error(model, X_tensor):
    """MSE po uzorku između originala i rekonstrukcije."""
    model.eval()
    with torch.no_grad():
        X_recon = model(X_tensor)
    X_np     = X_tensor.cpu().numpy()
    X_rec_np = X_recon.cpu().numpy()
    mse = np.mean(np.power(X_np - X_rec_np, 2), axis=1)
    return mse


test_mse  = reconstruction_error(autoencoder, X_test_t)

legit_mse = test_mse[y_test == 0]
fraud_mse = test_mse[y_test == 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(legit_mse, bins=100, alpha=0.6, color='#2ecc71', label='Legitimna', density=True)
axes[0].hist(fraud_mse, bins=100, alpha=0.6, color='#e74c3c', label='Fraud',     density=True)
axes[0].set_xlabel('Rekonstrukcijska Greška (MSE)')
axes[0].set_ylabel('Gustina')
axes[0].set_title('Distribucija Anomaly Score po Klasama')
axes[0].set_xlim(0, np.percentile(test_mse, 99.5))
axes[0].legend()

axes[1].hist(legit_mse, bins=100, alpha=0.6, color='#2ecc71', label='Legitimna', density=True)
axes[1].hist(fraud_mse, bins=100, alpha=0.6, color='#e74c3c', label='Fraud',     density=True)
axes[1].set_xlabel('Rekonstrukcijska Greška (MSE)')
axes[1].set_ylabel('Gustina (log)')
axes[1].set_title('Distribucija Anomaly Score (log osa)')
axes[1].set_yscale('log')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{RESULTS}05_anomaly_score_dist.png', bbox_inches='tight')
plt.show()

print(f'\nProsečna MSE — Legitimna: {legit_mse.mean():.4f}')
print(f'Prosečna MSE — Fraud:     {fraud_mse.mean():.4f}')
print(f'Separation ratio:          {fraud_mse.mean() / legit_mse.mean():.2f}x')

## 5. Određivanje Threshold-a i Evaluacija

**Kako odabrati threshold (prag odluke)?**

Za razliku od supervised modela, autoencoder nema direktno interpretirajući izlaz — anomaly score je kontinualna vrednost. Moramo odabrati threshold koji pretvara score u binarnu predikciju.

**Pristup:** Pretražujemo niz percentila val MSE distribucije (80-ti do 99.9-ti) kao potencijalne threshold-ove i biramo onaj koji maksimizuje F1 na validation setu.

> Koristimo percentile (umesto apsolutnih vrednosti) jer su robustniji prema različitim skalama grešaka.

In [ ]:
val_mse = reconstruction_error(autoencoder, X_val_t)

thresholds = np.percentile(val_mse, np.linspace(80, 99.9, 200))
f1s = [f1_score(y_val, (val_mse >= t).astype(int)) for t in thresholds]
best_t_idx = np.argmax(f1s)
best_t = thresholds[best_t_idx]

print(f'Optimalni threshold (percentila na val): {best_t:.4f}')
print(f'Odgovarajuća F1: {max(f1s):.4f}')

plt.figure(figsize=(9, 4))
plt.plot(thresholds, f1s, color='steelblue')
plt.axvline(best_t, color='red', linestyle='--', label=f'Optimalni threshold = {best_t:.3f}')
plt.xlabel('Threshold (MSE)')
plt.ylabel('F1 Score')
plt.title('F1 Score vs Anomaly Threshold')
plt.legend()
plt.savefig(f'{RESULTS}05_threshold_f1.png', bbox_inches='tight')
plt.show()

In [ ]:
# Finalna evaluacija na test setu
y_pred_ae = (test_mse >= best_t).astype(int)

roc_auc = roc_auc_score(y_test, test_mse)
pr_auc  = average_precision_score(y_test, test_mse)
f1      = f1_score(y_test, y_pred_ae)

print('=== AUTOENCODER — Finalna Evaluacija ===')
print(f'ROC-AUC: {roc_auc:.4f}')
print(f'PR-AUC:  {pr_auc:.4f}')
print(f'F1:      {f1:.4f}')
print()
print(classification_report(y_test, y_pred_ae, target_names=['Legitimna', 'Fraud']))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

cm = confusion_matrix(y_test, y_pred_ae)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
axes[0].set_title('Confusion Matrix')
axes[0].set_ylabel('Stvarno')
axes[0].set_xlabel('Predviđeno')

fpr, tpr, _ = roc_curve(y_test, test_mse)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.4f}')
axes[1].plot([0, 1], [0, 1], 'k--')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Kriva'); axes[1].legend()

prec, rec, _ = precision_recall_curve(y_test, test_mse)
axes[2].plot(rec, prec, color='steelblue', lw=2, label=f'PR-AUC = {pr_auc:.4f}')
axes[2].set_xlabel('Recall'); axes[2].set_ylabel('Precision')
axes[2].set_title('PR Kriva'); axes[2].legend()

plt.suptitle('Autoencoder Anomaly Detection', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS}05_ae_evaluation.png', bbox_inches='tight')
plt.show()

ae_results = {'title': 'Autoencoder', 'roc_auc': roc_auc, 'pr_auc': pr_auc,
              'f1': f1, 'threshold': best_t}
pd.DataFrame([ae_results]).to_csv(f'{RESULTS}05_ae_results.csv', index=False)

## 6. Vizualizacija Latentnog Prostora (t-SNE)

**t-SNE (t-distributed Stochastic Neighbor Embedding)** je nelinearna tehnika redukcije dimenzionalnosti koja dobro prikazuje klastere u visokodimenzionalnim podacima.

**Šta tražimo:** Ako je autoencoder naučio smislenu reprezentaciju, fraud transakcije bi trebale da formiraju zasebne klastere u latentnom prostoru — odvojene od legitimnih transakcija. Ovo nam daje vizuelni uvid u kvalitet naučenih reprezentacija.

**Napomena:** t-SNE je računski zahtevno, pa uzimamo random sample od 3000 uzoraka.

Koristimo samo **encoder deo** autoencodera za ekstrakciju latentnih vektora — ovo je prednost modularnog dizajna.

In [ ]:
n_sample = 3000
idx      = np.random.choice(len(X_test), n_sample, replace=False)
X_sample = torch.tensor(X_test[idx], dtype=torch.float32).to(DEVICE)
y_sample = y_test[idx]

# Enkodujemo u latentni prostor koristeći samo Encoder deo
autoencoder.eval()
with torch.no_grad():
    latent = autoencoder.encoder(X_sample).cpu().numpy()

print(f'Latentni prostor: {latent.shape}  (8-dimenzionalni)')
print('Pokrećem t-SNE (može trajati par minuta)...')

tsne = TSNE(n_components=2, random_state=RANDOM_SEED, perplexity=30, n_iter=1000)
latent_2d = tsne.fit_transform(latent)

fig, ax = plt.subplots(figsize=(10, 7))

# Crtamo legitimne transakcije (mali, polu-transparentni)
legit_mask = (y_sample == 0)
ax.scatter(latent_2d[legit_mask, 0], latent_2d[legit_mask, 1],
           c='#2ecc71', s=8, alpha=0.3, label=f'Legitimna (n={legit_mask.sum()})')

# Crtamo fraud transakcije (veće, vidljivije)
fraud_mask = (y_sample == 1)
ax.scatter(latent_2d[fraud_mask, 0], latent_2d[fraud_mask, 1],
           c='#e74c3c', s=60, alpha=0.9, label=f'Fraud (n={fraud_mask.sum()})',
           edgecolors='darkred', linewidths=0.5)

ax.legend(fontsize=12)
ax.set_title('t-SNE vizualizacija Latentnog Prostora Autoencodera',
             fontsize=13, fontweight='bold')
ax.set_xlabel('t-SNE Dimenzija 1')
ax.set_ylabel('t-SNE Dimenzija 2')

plt.tight_layout()
plt.savefig(f'{RESULTS}05_tsne_latent.png', bbox_inches='tight')
plt.show()

print('\n→ Sledeći notebook: 06_evaluation.ipynb')